# Results for model: deepseek-ai/DeepSeek-V3

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

quantile_df = df.group_by('date_id').agg(
    pl.col('feature_00').quantile(0.85).alias('top_quantile')
).sort('date_id')

rolling_quantile = quantile_df.with_columns(
    pl.col('top_quantile').rolling_mean(window_size=5, min_periods=1).alias('rolling_top_quantile')
)

df = df.join(rolling_quantile, on='date_id')
df = df.with_columns(
    (pl.col('feature_00') > pl.col('rolling_top_quantile')).cast(pl.Int8).alias('feature_00_top_quantile')
)

features = [col for col in df.columns if col.startswith('feature_') and col != 'responder_6']
features.append('feature_00_top_quantile')

X = df.select(features).to_numpy()
y = df.select('responder_6').to_numpy().ravel()

model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
model.fit(X, y)